In [11]:
using DataFrames, Distributions, CSV, Turing, Optim, StatsBase, StatsFuns
include("measures.jl")

# Read the vitamin D dataset
data = CSV.read("../../data/vitd/nhanes.csv", DataFrame);

# Look at the first few rows of the data
first(data, 5)

Row,BMXBMI,RIAGENDR,RIDAGEYR,LBXVIDMS,INDFMMPI,DPQ020,SMQ040
,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,27.8,1.0,62.0,76.1,4.14,0.0,3.0
2,30.8,1.0,53.0,56.5,0.0,0.0,1.0
3,28.8,1.0,78.0,87.5,1.81,0.0,3.0
4,28.0,1.0,22.0,47.2,2.98,0.0,2.0
5,27.6,1.0,46.0,44.5,1.73,0.0,1.0


In [13]:
data[!, :"RIAGENDR"] = data[!, :"RIAGENDR"] .- 1;

1628-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 1.0
 1.0
 1.0
 1.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 1.0
 1.0
 1.0
 0.0
 1.0

In [2]:
#= Convert categorical data to integers =#

data[!, :"RIAGENDR"] = convert(Array{Int}, data[!, :"RIAGENDR"])
data[!, "DPQ020"] = convert(Array{Int}, data[!, "DPQ020"])
data[!, :"SMQ040"] = convert(Array{Int}, data[!, :"SMQ040"]);

# Get variables
bmi = data[!, :"BMXBMI"]
gender = data[!, :"RIAGENDR"]
age = data[!, :"RIDAGEYR"]
vitd = data[!, :"LBXVIDMS"]
poverty = data[!, :"INDFMMPI"]
depression = data[!, :"DPQ020"]
smoking = data[!, :"SMQ040"];

In [3]:
first(data, 5)

Row,BMXBMI,RIAGENDR,RIDAGEYR,LBXVIDMS,INDFMMPI,DPQ020,SMQ040
,Float64,Int64,Float64,Float64,Float64,Int64,Int64
1,27.8,1,62.0,76.1,4.14,0,3
2,30.8,1,53.0,56.5,0.0,0,1
3,28.8,1,78.0,87.5,1.81,0,3
4,28.0,1,22.0,47.2,2.98,0,2
5,27.6,1,46.0,44.5,1.73,0,1


In [4]:
# Turing Model NO BIDIRECTION

@model function depression_model(bmi, gender, age, vitd, poverty, depression, smoking)
    n = length(bmi)

    # Define the linear models
    for i in 1:n
        gender[i] ~ Bernoulli(0.4)
        age_dist1 = Normal(36, 10)
        age_dist2 = Normal(62, 10)
        age[i] ~ MixtureModel([age_dist1, age_dist2], [0.4, 0.6])
        smoking[i] ~ Categorical([0.35, 0.12, 0.53])
    
        bmi[i] ~ LogNormal(3.39196 + gender[i] * 3.39196 + age[i] * 0.000118077, 0.22)
        poverty_dist1 = Normal(2.07895, 2)
        poverty_dist2 = Normal(1.05, 0.001)
        poverty[i] ~ MixtureModel([poverty_dist1, poverty_dist2], [0.5, 0.5])

       # Latent variable for depression
       depression[i] ~ LogNormal(0.125138 + -3.472 * vitd[i] * (10^(-5)) + 0.000206753 * age[i]
       + 0.0358828 * gender[i] + -0.0408787 * smoking[i], 0.6)

       # Round and clamp the depression levels
       depression[i] = clamp(round(Int, depression[i]), 1, 4)

        vitd[i] ~ LogNormal(3.32057 + bmi[i] * 0.00378878 + age[i] * 0.00671247 + poverty[i] * 0.0611797 + gender[i] * 0.125546 + smoking[i] * 0.0993063, 0.42)
    end
end

depression_model (generic function with 2 methods)

In [5]:
cont_sampler = HMC(0.01, 10, :bmi, :age, :poverty, :sedentary, :vitd, :depression)
disc_sampler = PG(10, :gender, :smoking, :depression)
sampler = Gibbs(cont_sampler, disc_sampler)

Gibbs{(:bmi, :age, :poverty, :sedentary, :vitd, :depression, :gender, :smoking), Tuple{HMC{Turing.Essential.ForwardDiffAD{0}, (:bmi, :age, :poverty, :sedentary, :vitd, :depression), AdvancedHMC.UnitEuclideanMetric}, PG{(:gender, :smoking, :depression), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}}}((HMC{Turing.Essential.ForwardDiffAD{0}, (:bmi, :age, :poverty, :sedentary, :vitd, :depression), AdvancedHMC.UnitEuclideanMetric}(0.01, 10), PG{(:gender, :smoking, :depression), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}(10, AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}(AdvancedPS.resample_systematic, 0.5))))

In [6]:
model = depression_model(bmi, gender, age, vitd, poverty, repeat([missing], length(bmi)), smoking)
#chain = sample(model, sampler, MCMCThreads(), 500, 6)
chain = sample(model, sampler, 50)

Sampling   0%|                                          |  ETA: N/A
Sampling   2%|▉                                         |  ETA: 0:15:42
Sampling   4%|█▋                                        |  ETA: 0:13:43
Sampling   6%|██▌                                       |  ETA: 0:12:13
Sampling   8%|███▍                                      |  ETA: 0:11:21
Sampling  10%|████▎                                     |  ETA: 0:10:45
Sampling  12%|█████                                     |  ETA: 0:10:13
Sampling  14%|█████▉                                    |  ETA: 0:09:56
Sampling  16%|██████▊                                   |  ETA: 0:09:39
Sampling  18%|███████▌                                  |  ETA: 0:09:22
Sampling  20%|████████▍                                 |  ETA: 0:09:07
Sampling  22%|█████████▎                                |  ETA: 0:08:52
Sampling  24%|██████████▏                               |  ETA: 0:08:37
Sampling  26%|██████████▉                               |  ETA: 0:08

Chains MCMC chain (50×1629×1 Array{Float64, 3}):

Iterations        = 1:1:50
Number of chains  = 1
Samples per chain = 50
Wall duration     = 642.33 seconds
Compute duration  = 642.33 seconds
parameters        = depression[1], depression[2], depression[3], depression[4], depression[5], depression[6], depression[7], depression[8], depression[9], depression[10], depression[11], depression[12], depression[13], depression[14], depression[15], depression[16], depression[17], depression[18], depression[19], depression[20], depression[21], depression[22], depression[23], depression[24], depression[25], depression[26], depression[27], depression[28], depression[29], depression[30], depression[31], depression[32], depression[33], depression[34], depression[35], depression[36], depression[37], depression[38], depression[39], depression[40], depression[41], depression[42], depression[43], depression[44], depression[45], depression[46], depression[47], depression[48], depression[49], depression[50

In [10]:
mean(chain["depression[2]"])

9.115257435630753